In [1]:
!pip install pandas numpy scikit-learn xgboost optuna shap matplotlib seaborn

# Phase 9-12: ML Models, Tuning & Comparison

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
try:
    from xgboost import XGBRegressor
except BaseException as e:
    print(f"Skipping XGBoost due to error: {e}")
    print('XGBoost not available.')
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna

feat_df = pd.read_csv('/Users/anshul/Documents/AI/data/raw/MOSFET/engineered_features.csv')

X = feat_df[['Value_0h', 'Value_2h', 'Delta_Value', 'Early_Slope']]
y = feat_df['Target_Value']
groups = feat_df['Run_ID']

# Since we only have 2 runs, KFold with n_splits=2
gkf = GroupKFold(n_splits=2)

models = {
    'Persistence (Baseline)': None,
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}
try:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(random_state=42)
except BaseException:
    print("Skipping XGBoost (not available on this system due to missing OpenMP/libomp)")

results = []

for name, model in models.items():
    cv_mae = []
    cv_rmse = []
    cv_r2 = []
    
    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        if name == 'Persistence (Baseline)':
            # Predict the 2h value
            preds = X_test['Value_2h']
        else:
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            
        cv_mae.append(mean_absolute_error(y_test, preds))
        cv_rmse.append(np.sqrt(mean_squared_error(y_test, preds)))
        # R2 might be NaN for single sample, handle it gracefully
        if len(y_test) > 1:
            cv_r2.append(r2_score(y_test, preds))
        else:
            cv_r2.append(np.nan)
            
    results.append({
        'Model': name,
        'CV MAE': np.mean(cv_mae),
        'CV RMSE': np.mean(cv_rmse),
        'CV R²': np.nanmean(cv_r2) if not np.isnan(cv_r2).all() else np.nan,
        'Std MAE': np.std(cv_mae)
    })

res_df = pd.DataFrame(results)
print(res_df)

# Hyperparameter Tuning with Optuna (Example for XGBoost)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    mae_list = []
    for train_idx, test_idx in gkf.split(X, y, groups):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        mae_list.append(mean_absolute_error(y.iloc[test_idx], preds))
    return np.mean(mae_list)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)
print("Best Params:", study.best_params)


/Users/anshul/Documents/AI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-31 22:36:59,435] A new study created in memory with name: no-name-435326ca-c129-4f35-86ca-02bd43d2032d
[I 2026-08-31 22:36:59,454] Trial 0 finished with value: 1.6547009064821319 and parameters: {'n_estimators': 99, 'learning_rate': 0.01475198410548863, 'max_depth': 4}. Best is trial 0 with value: 1.6547009064821319.
[I 2026-08-31 22:36:59,465] Trial 1 finished with value: 1.6547009064821319 and parameters: {'n_estimators': 50, 'learning_rate': 0.006699981702358121, 'max_depth': 4}. Best is trial 0 with value: 1.6547009064821319.
[I 2026-08-31 22:36:59,476] Trial 2 finished with value: 1.6547009064821319 and parameters: {'n_estimators': 48, 'learning_rate': 0.0020869239949638744, 'max_depth': 3}. Best is trial 0 

Skipping XGBoost due to error: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/anshul/Documents/AI/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <09B43001-8527-34F6-AAE4-B97DA8A6CBC0> /Users/anshul/Documents/AI/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/